WEB SCRAPING


In [2]:
pip install google-play-scraper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.7 MB/s eta 0:00:00


In [3]:
import pandas as pd
from google_play_scraper import reviews, Sort
import re
# ENTER GOOGLE PLAY URL
url = input("Enter Google Play URL: ").strip()
# Extract the app ID from the URL
match = re.search(r"id=([A-Za-z0-9._-]+)", url)
if not match:
    raise ValueError("Invalid Google Play URL. Could not find app ID.")
app_id = match.group(1)

print(f"Detected App ID: {app_id}")
# Fetch reviews (newest 200 reviews)
result, _ = reviews(
    app_id,
    lang="en",
    country="us",
    sort=Sort.NEWEST,
    count=100
)
# Convert to pandas DataFrame
df = pd.DataFrame(result)
# Save to CSV
output_file = f"{app_id}_reviews.csv"
df.to_csv(output_file, index=False)

print(f"Saved {len(df)} reviews to {output_file}")  #https://play.google.com/store/apps/details?id=com.kiloo.subwaysurf

Enter Google Play URL: https://play.google.com/store/apps/details?id=com.kiloo.subwaysurf
Detected App ID: com.kiloo.subwaysurf
Saved 100 reviews to com.kiloo.subwaysurf_reviews.csv


In [5]:
from google.colab import files
uploaded = files.upload()

Saving com.kiloo.subwaysurf_reviews.csv to com.kiloo.subwaysurf_reviews.csv


In [8]:


import pandas as pd
# ---------- 0) Load ----------
df = pd.read_csv("com.kiloo.subwaysurf_reviews.csv")
print("After load:", df.shape)
df.head()


After load: (100, 11)


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,1e60bda0-059e-4650-a3e1-11b30051f242,Leo Diaz,https://play-lh.googleusercontent.com/a-/ALV-U...,it doesn't keep your stuff when you delete and...,1,0,NaN,2026-03-17 00:54:52,Thank you for sharing your experience. Losing ...,2026-03-17 02:35:10,NaN
1,415c6293-5ab6-4db6-8a71-93c9c00ef898,Ryan Klutts,https://play-lh.googleusercontent.com/a-/ALV-U...,fun and play offline,5,0,3.60.0,2026-03-17 00:35:12,NaN,NaN,3.60.0
2,d1fdcb36-4b97-4fde-9e57-fb0d813cad09,ellen yeboah,https://play-lh.googleusercontent.com/a/ACg8oc...,cool,5,0,NaN,2026-03-16 23:37:53,NaN,NaN,NaN
3,2a82c3be-e067-4624-b220-f1dc8b23fd3b,Noah Tarpley,https://play-lh.googleusercontent.com/a/ACg8oc...,Amazing.,5,0,3.60.0,2026-03-16 23:19:20,NaN,NaN,3.60.0
4,a48bfa0a-5a36-48b7-b105-ba603b8106cf,Aisha Sumaiya,https://play-lh.googleusercontent.com/a/ACg8oc...,no words,5,0,3.49.0,2026-03-16 22:54:12,NaN,NaN,3.49.0


PRE-PROCCESSING

In [9]:

# 1. Drop the userImage column (not useful)
df = df.drop(columns=["userImage"])

# 2. Reassign reviewId values from 1 to N
df["reviewId"] = range(1, len(df) + 1)

# 3. Clean userName:
#    - strip spaces
#    - convert to title case (first letter capital)
df["userName"] = df["userName"].astype(str).str.strip().str.title()
df.head()


,reviewId,userName,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,1,Leo Diaz,it doesn't keep your stuff when you delete and...,1,0,NaN,2026-03-17 00:54:52,Thank you for sharing your experience. Losing ...,2026-03-17 02:35:10,NaN
1,2,Ryan Klutts,fun and play offline,5,0,3.60.0,2026-03-17 00:35:12,NaN,NaN,3.60.0
2,3,Ellen Yeboah,cool,5,0,NaN,2026-03-16 23:37:53,NaN,NaN,NaN
3,4,Noah Tarpley,Amazing.,5,0,3.60.0,2026-03-16 23:19:20,NaN,NaN,3.60.0
4,5,Aisha Sumaiya,no words,5,0,3.49.0,2026-03-16 22:54:12,NaN,NaN,3.49.0


In [10]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 13.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=202ab8a3a5ca32d043c6c761f19bd558e12d6a574f3b41aabc5940c9ed632469
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [11]:

import re
from langdetect import detect, LangDetectException



In [12]:


# -------- 1. Detect language --------
def detect_language(text):
    try:
        return detect(str(text))
    except LangDetectException:
        return "unknown"

df["lang"] = df["content"].apply(detect_language)

# -------- 2. Keep only English content --------
df = df[df["lang"] == "en"]

# -------- 3. Clean English text --------
def clean_text(text):
    text = str(text)

    # Remove emojis
    text = re.sub(r"[^\w\s.,!?']", "", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove special characters except punctuation
    text = re.sub(r"[^A-Za-z0-9\s.,!?']", "", text)

    # Lowercase
    text = text.lower()

    # Remove extra spaces
    text = " ".join(text.split())

    return text

df["content"] = df["content"].apply(clean_text)

# Drop language column
df = df.drop(columns=["lang"])





In [13]:
df.head()

,reviewId,userName,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,1,Leo Diaz,it doesn't keep your stuff when you delete and...,1,0,NaN,2026-03-17 00:54:52,Thank you for sharing your experience. Losing ...,2026-03-17 02:35:10,NaN
1,2,Ryan Klutts,fun and play offline,5,0,3.60.0,2026-03-17 00:35:12,NaN,NaN,3.60.0
2,3,Ellen Yeboah,cool,5,0,NaN,2026-03-16 23:37:53,NaN,NaN,NaN
5,6,Ahmadabalrahman Ahmad,it's good,5,0,3.60.0,2026-03-16 22:44:36,NaN,NaN,3.60.0
8,9,Ramlat Kabir Wada,i love this game it deserves 5 stars and works...,5,0,NaN,2026-03-16 21:04:13,NaN,NaN,NaN


In [14]:

import numpy as np
#  Rename score → rating
df = df.rename(columns={"score": "rating"})

#  Clean the rating column
# Convert to numeric (anything invalid becomes NaN)
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

# Replace missing ratings with median
median_rating = df["rating"].median()
df["rating"] = df["rating"].fillna(median_rating)

# Ensure all ratings are within the valid range [1–5]
df["rating"] = df["rating"].clip(lower=1, upper=5)

# Enforce integer star ratings
df["rating"] = df["rating"].round().astype(int)


# Clean the thumbsUpCount column
# Convert to numeric
df["thumbsUpCount"] = pd.to_numeric(df["thumbsUpCount"], errors="coerce")

# Replace NaN with 0
df["thumbsUpCount"] = df["thumbsUpCount"].fillna(0)

# Floor decimals and remove negative values
df["thumbsUpCount"] = np.floor(df["thumbsUpCount"]).astype(int)
df["thumbsUpCount"] = df["thumbsUpCount"].clip(lower=0)



In [15]:
df.head()

,reviewId,userName,content,rating,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,1,Leo Diaz,it doesn't keep your stuff when you delete and...,1,0,NaN,2026-03-17 00:54:52,Thank you for sharing your experience. Losing ...,2026-03-17 02:35:10,NaN
1,2,Ryan Klutts,fun and play offline,5,0,3.60.0,2026-03-17 00:35:12,NaN,NaN,3.60.0
2,3,Ellen Yeboah,cool,5,0,NaN,2026-03-16 23:37:53,NaN,NaN,NaN
5,6,Ahmadabalrahman Ahmad,it's good,5,0,3.60.0,2026-03-16 22:44:36,NaN,NaN,3.60.0
8,9,Ramlat Kabir Wada,i love this game it deserves 5 stars and works...,5,0,NaN,2026-03-16 21:04:13,NaN,NaN,NaN


In [16]:
#integerating 	reviewCreatedVersion and appVersion
def normalize_version(v):
    if pd.isna(v):
        return np.nan
    v = str(v).strip()
    v = re.sub(r"[^0-9.]", "", v)
    v = re.sub(r"\.+", ".", v).strip(".")
    return v if v else np.nan

# Normalize both columns
df["reviewCreatedVersion"] = df["reviewCreatedVersion"].apply(normalize_version)
df["appVersion"] = df["appVersion"].apply(normalize_version)

# Create merged column
df["rated_app_version"] = df["reviewCreatedVersion"].fillna(df["appVersion"])
df["rated_app_version"] = df["rated_app_version"].fillna("unknown")


df = df.drop(columns=["reviewCreatedVersion", "appVersion"])

df.head(10)

,reviewId,userName,content,rating,thumbsUpCount,at,replyContent,repliedAt,rated_app_version
0,1,Leo Diaz,it doesn't keep your stuff when you delete and...,1,0,2026-03-17 00:54:52,Thank you for sharing your experience. Losing ...,2026-03-17 02:35:10,unknown
1,2,Ryan Klutts,fun and play offline,5,0,2026-03-17 00:35:12,NaN,NaN,3.60.0
2,3,Ellen Yeboah,cool,5,0,2026-03-16 23:37:53,NaN,NaN,unknown
5,6,Ahmadabalrahman Ahmad,it's good,5,0,2026-03-16 22:44:36,NaN,NaN,3.60.0
8,9,Ramlat Kabir Wada,i love this game it deserves 5 stars and works...,5,0,2026-03-16 21:04:13,NaN,NaN,unknown
9,10,Tomilola Aribo,"this game is a straight 10, to be honest. alwa...",5,0,2026-03-16 21:00:34,NaN,NaN,3.60.0
11,12,Jace Dunton,be your self and play subway surfers.,4,0,2026-03-16 20:40:17,NaN,NaN,unknown
12,13,Chloe Shore,works without internet,5,0,2026-03-16 20:31:23,NaN,NaN,3.60.0
14,15,Summer,really fun,5,0,2026-03-16 20:19:56,NaN,NaN,3.60.0
15,16,Adewale Basira,this is a seriously game and very helpful game,5,0,2026-03-16 20:12:36,NaN,NaN,unknown


In [17]:

# Drop repliedat and at
columns_to_drop = ["repliedAt", "at"]
df = df.drop(columns=[c for c in columns_to_drop if c in df.columns])

# 2) Clean replyContent
def clean_reply(text):
    # If NaN -> return NaN (no cleaning)
    if pd.isna(text):
        return text

    text = str(text)

    # Remove emojis (keep only letters, numbers, punctuation)
    text = re.sub(r"[^\w\s.,!?']", "", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove unwanted characters
    text = re.sub(r"[^A-Za-z0-9\s.,!?']", "", text)

    # Lowercase
    text = text.lower()

    # Remove extra spaces
    text = " ".join(text.split())

    return text

# Apply only if column exists
if "replyContent" in df.columns:
    df["replyContent"] = df["replyContent"].apply(clean_reply)



In [18]:
df.head(10)

,reviewId,userName,content,rating,thumbsUpCount,replyContent,rated_app_version
0,1,Leo Diaz,it doesn't keep your stuff when you delete and...,1,0,thank you for sharing your experience. losing ...,unknown
1,2,Ryan Klutts,fun and play offline,5,0,NaN,3.60.0
2,3,Ellen Yeboah,cool,5,0,NaN,unknown
5,6,Ahmadabalrahman Ahmad,it's good,5,0,NaN,3.60.0
8,9,Ramlat Kabir Wada,i love this game it deserves 5 stars and works...,5,0,NaN,unknown
9,10,Tomilola Aribo,"this game is a straight 10, to be honest. alwa...",5,0,NaN,3.60.0
11,12,Jace Dunton,be your self and play subway surfers.,4,0,NaN,unknown
12,13,Chloe Shore,works without internet,5,0,NaN,3.60.0
14,15,Summer,really fun,5,0,NaN,3.60.0
15,16,Adewale Basira,this is a seriously game and very helpful game,5,0,NaN,unknown


In [19]:
# Save cleaned dataset to CSV
df.to_csv("cleaned_subway_surfers_reviews.csv", index=False)

print("✅ Cleaning completed")
print("✅ CSV file saved: cleaned_subway_surfers_reviews.csv")
print("Dataset shape:", df.shape)

# Show first rows
df.head(10)

✅ Cleaning completed
✅ CSV file saved: cleaned_subway_surfers_reviews.csv
Dataset shape: (63, 7)


,reviewId,userName,content,rating,thumbsUpCount,replyContent,rated_app_version
0,1,Leo Diaz,it doesn't keep your stuff when you delete and...,1,0,thank you for sharing your experience. losing ...,unknown
1,2,Ryan Klutts,fun and play offline,5,0,NaN,3.60.0
2,3,Ellen Yeboah,cool,5,0,NaN,unknown
5,6,Ahmadabalrahman Ahmad,it's good,5,0,NaN,3.60.0
8,9,Ramlat Kabir Wada,i love this game it deserves 5 stars and works...,5,0,NaN,unknown
9,10,Tomilola Aribo,"this game is a straight 10, to be honest. alwa...",5,0,NaN,3.60.0
11,12,Jace Dunton,be your self and play subway surfers.,4,0,NaN,unknown
12,13,Chloe Shore,works without internet,5,0,NaN,3.60.0
14,15,Summer,really fun,5,0,NaN,3.60.0
15,16,Adewale Basira,this is a seriously game and very helpful game,5,0,NaN,unknown


In [20]:
from google.colab import files
uploaded = files.upload()

Saving cleaned_subway_surfers_reviews.csv to cleaned_subway_surfers_reviews (1).csv


FIRST ANANNOTATOR (RATING COLUMN)

In [21]:
import pandas as pd

df = pd.read_csv("cleaned_subway_surfers_reviews.csv")

df.head()

,reviewId,userName,content,rating,thumbsUpCount,replyContent,rated_app_version
0,1,Leo Diaz,it doesn't keep your stuff when you delete and...,1,0,thank you for sharing your experience. losing ...,unknown
1,2,Ryan Klutts,fun and play offline,5,0,NaN,3.60.0
2,3,Ellen Yeboah,cool,5,0,NaN,unknown
3,6,Ahmadabalrahman Ahmad,it's good,5,0,NaN,3.60.0
4,9,Ramlat Kabir Wada,i love this game it deserves 5 stars and works...,5,0,NaN,unknown


In [22]:
SENTIMENTS = ["Positive", "Neutral", "Negative"]
def rating_to_sentiment(rating):
    if rating >= 4:
        return "Positive"
    elif rating == 3:
        return "Neutral"
    else:
        return "Negative"

df["annotator_1_sentiment"] = df["rating"].apply(rating_to_sentiment)

df[["rating", "annotator_1_sentiment"]].head()

,rating,annotator_1_sentiment
0,1,Negative
1,5,Positive
2,5,Positive
3,5,Positive
4,5,Positive


VADER(2ND ANNATATOR)

In [23]:
!pip install vaderSentiment


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 412.8 kB/s eta 0:00:00


In [24]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [25]:
#initialize
vader = SentimentIntensityAnalyzer()

In [26]:
#testing
vader.polarity_scores("I really love this game!")


{'neg': 0.0, 'neu': 0.455, 'pos': 0.545, 'compound': 0.6989}

In [27]:
#create function
def vader_sentiment(text):
    scores = vader.polarity_scores(str(text))
    compound = scores["compound"]

    if compound >= 0.05:
        return "Positive"
    elif compound <= -0.05:
        return "Negative"
    else:
        return "Neutral"

In [28]:
#apply
df["vader_sentiment"] = df["content"].apply(vader_sentiment)

In [29]:
#save to df
df["vader_compound"] = df["content"].apply(
    lambda x: vader.polarity_scores(str(x))["compound"]
)

In [30]:
df[["content", "vader_sentiment", "vader_compound"]].head(10)

,content,vader_sentiment,vader_compound
0,it doesn't keep your stuff when you delete and...,Negative,-0.5719
1,fun and play offline,Positive,0.6369
2,cool,Positive,0.3182
3,it's good,Positive,0.4404
4,i love this game it deserves 5 stars and works...,Positive,0.6369
5,"this game is a straight 10, to be honest. alwa...",Positive,0.8439
6,be your self and play subway surfers.,Positive,0.3400
7,works without internet,Neutral,0.0000
8,really fun,Positive,0.5563
9,this is a seriously game and very helpful game,Positive,0.3384


In [31]:
df.head(10)

,reviewId,userName,content,rating,thumbsUpCount,replyContent,rated_app_version,annotator_1_sentiment,vader_sentiment,vader_compound
0,1,Leo Diaz,it doesn't keep your stuff when you delete and...,1,0,thank you for sharing your experience. losing ...,unknown,Negative,Negative,-0.5719
1,2,Ryan Klutts,fun and play offline,5,0,NaN,3.60.0,Positive,Positive,0.6369
2,3,Ellen Yeboah,cool,5,0,NaN,unknown,Positive,Positive,0.3182
3,6,Ahmadabalrahman Ahmad,it's good,5,0,NaN,3.60.0,Positive,Positive,0.4404
4,9,Ramlat Kabir Wada,i love this game it deserves 5 stars and works...,5,0,NaN,unknown,Positive,Positive,0.6369
5,10,Tomilola Aribo,"this game is a straight 10, to be honest. alwa...",5,0,NaN,3.60.0,Positive,Positive,0.8439
6,12,Jace Dunton,be your self and play subway surfers.,4,0,NaN,unknown,Positive,Positive,0.3400
7,13,Chloe Shore,works without internet,5,0,NaN,3.60.0,Positive,Neutral,0.0000
8,15,Summer,really fun,5,0,NaN,3.60.0,Positive,Positive,0.5563
9,16,Adewale Basira,this is a seriously game and very helpful game,5,0,NaN,unknown,Positive,Positive,0.3384


AFFIN(3RD ANNATATOR)

In [32]:
!pip install afinn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 740.4 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for afinn: filename=afinn-0.1-py3-none-any.whl size=53431 sha256=30febb5957fa3f1b3a58993a7e06b52194fc4d0a3994b7a4245335ce6a16c3f3
  Stored in directory: /root/.cache/pip/wheels/f9/72/27/74994e77200dae3d6aea2b546264500cee21f738c51241320b
Successfully built afinn


In [33]:
import re
from afinn import Afinn

In [34]:
afinn = Afinn()

In [35]:
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return text.split()  #tokens

In [36]:
negation_words = {
    "not", "no", "never", "none", "nothing",
    "dont", "didnt", "doesnt", "cant", "wont",
    "isnt", "wasnt", "werent", "arent",
    "couldnt", "shouldnt", "wouldnt",
    "without", "lack", "lacking", "missing", "unable"

}

In [37]:

#iterates through each token, sums sentiment scores,and reverses sentiment polarity when negation words are detected.

def afinn_with_negation(text):
    tokens = preprocess(text)
    score = 0
    i = 0

    while i < len(tokens):
        word = tokens[i]

        # If current word is a negation
        if word in negation_words and i + 1 < len(tokens):
            next_word = tokens[i + 1]
            score -= afinn.score(next_word)  # flip polarity
            i += 2
        else:
            score += afinn.score(word)
            i += 1

    return score

In [38]:
def score_to_sentiment(score):
    if score > 0:
        return "Positive"
    elif score < 0:
        return "Negative"
    else:
        return "Neutral"

In [39]:
df["afinn_score"] = df["content"].apply(afinn_with_negation)
df["afinn_sentiment"] = df["afinn_score"].apply(score_to_sentiment)

In [40]:
df[["content", "afinn_score", "afinn_sentiment"]].head(10)

,content,afinn_score,afinn_sentiment
0,it doesn't keep your stuff when you delete and...,-3.0,Negative
1,fun and play offline,3.0,Positive
2,cool,1.0,Positive
3,it's good,3.0,Positive
4,i love this game it deserves 5 stars and works...,3.0,Positive
5,"this game is a straight 10, to be honest. alwa...",2.0,Positive
6,be your self and play subway surfers.,0.0,Neutral
7,works without internet,0.0,Neutral
8,really fun,4.0,Positive
9,this is a seriously game and very helpful game,2.0,Positive


In [41]:
df.to_csv("afinn,vader,1annator.csv", index=False)

In [42]:
df.head(10)

,reviewId,userName,content,rating,thumbsUpCount,replyContent,rated_app_version,annotator_1_sentiment,vader_sentiment,vader_compound,afinn_score,afinn_sentiment
0,1,Leo Diaz,it doesn't keep your stuff when you delete and...,1,0,thank you for sharing your experience. losing ...,unknown,Negative,Negative,-0.5719,-3.0,Negative
1,2,Ryan Klutts,fun and play offline,5,0,NaN,3.60.0,Positive,Positive,0.6369,3.0,Positive
2,3,Ellen Yeboah,cool,5,0,NaN,unknown,Positive,Positive,0.3182,1.0,Positive
3,6,Ahmadabalrahman Ahmad,it's good,5,0,NaN,3.60.0,Positive,Positive,0.4404,3.0,Positive
4,9,Ramlat Kabir Wada,i love this game it deserves 5 stars and works...,5,0,NaN,unknown,Positive,Positive,0.6369,3.0,Positive
5,10,Tomilola Aribo,"this game is a straight 10, to be honest. alwa...",5,0,NaN,3.60.0,Positive,Positive,0.8439,2.0,Positive
6,12,Jace Dunton,be your self and play subway surfers.,4,0,NaN,unknown,Positive,Positive,0.3400,0.0,Neutral
7,13,Chloe Shore,works without internet,5,0,NaN,3.60.0,Positive,Neutral,0.0000,0.0,Neutral
8,15,Summer,really fun,5,0,NaN,3.60.0,Positive,Positive,0.5563,4.0,Positive
9,16,Adewale Basira,this is a seriously game and very helpful game,5,0,NaN,unknown,Positive,Positive,0.3384,2.0,Positive


MAJORITY VOTING FOR GROUNF TRUTH LABEL

In [51]:
# majority voting across Rating, VADER, and AFINN sentiment labels

from collections import Counter

def majority_vote(row):
    votes = [
        row["annotator_1_sentiment"],
        row["vader_sentiment"],
        row["afinn_sentiment"]
    ]

    vote_counts = Counter(votes)
    most_common_label, count = vote_counts.most_common(1)[0]

    # If at least two annotators agree, use that label
    if count >= 2:
        return most_common_label
    else:
        # No agreement → Neutral
        return "Neutral"

In [54]:

# Create the final ground-truth sentiment column

df["ground_truth_sentiment"] = df.apply(majority_vote, axis=1)

df[[
    "annotator_1_sentiment",
    "vader_sentiment",
    "afinn_sentiment",
    "ground_truth_sentiment"
]].head(10)

,annotator_1_sentiment,vader_sentiment,afinn_sentiment,ground_truth_sentiment
0,Negative,Negative,Negative,Negative
1,Positive,Positive,Positive,Positive
2,Positive,Positive,Positive,Positive
3,Positive,Positive,Positive,Positive
4,Positive,Positive,Positive,Positive
5,Positive,Positive,Positive,Positive
6,Positive,Positive,Neutral,Positive
7,Positive,Neutral,Neutral,Neutral
8,Positive,Positive,Positive,Positive
9,Positive,Positive,Positive,Positive


In [55]:
# Check distribution of ground truth labels

df["ground_truth_sentiment"].value_counts()

,count
ground_truth_sentiment,
Positive,48
Neutral,10
Negative,5


In [56]:


df.to_csv("final_ground_truth_dataset.csv", index=False)

VALIDATING USING COHEN KAPPA

In [43]:

import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import cohen_kappa_score

df = pd.read_csv("afinn,vader,1annator.csv")

df.head()


,reviewId,userName,content,rating,thumbsUpCount,replyContent,rated_app_version,annotator_1_sentiment,vader_sentiment,vader_compound,afinn_score,afinn_sentiment
0,1,Leo Diaz,it doesn't keep your stuff when you delete and...,1,0,thank you for sharing your experience. losing ...,unknown,Negative,Negative,-0.5719,-3.0,Negative
1,2,Ryan Klutts,fun and play offline,5,0,NaN,3.60.0,Positive,Positive,0.6369,3.0,Positive
2,3,Ellen Yeboah,cool,5,0,NaN,unknown,Positive,Positive,0.3182,1.0,Positive
3,6,Ahmadabalrahman Ahmad,it's good,5,0,NaN,3.60.0,Positive,Positive,0.4404,3.0,Positive
4,9,Ramlat Kabir Wada,i love this game it deserves 5 stars and works...,5,0,NaN,unknown,Positive,Positive,0.6369,3.0,Positive


In [44]:

rating_labels = df["annotator_1_sentiment"]
vader_labels = df["vader_sentiment"]
afinn_labels = df["afinn_sentiment"]


In [46]:

#  converts categorical sentiment labels into numerical values for Kappa calculation.

encoder = LabelEncoder()

rating_encoded = encoder.fit_transform(rating_labels)
vader_encoded = encoder.transform(vader_labels)
afinn_encoded = encoder.transform(afinn_labels)


In [47]:
#  Cohen’s Kappa scores between different sentiment annotators.

kappa_rating_vader = cohen_kappa_score(rating_encoded, vader_encoded)
kappa_rating_afinn = cohen_kappa_score(rating_encoded, afinn_encoded)
kappa_vader_afinn = cohen_kappa_score(vader_encoded, afinn_encoded)

print("Cohen’s Kappa (Rating vs VADER):", round(kappa_rating_vader, 3))
print("Cohen’s Kappa (Rating vs AFINN):", round(kappa_rating_afinn, 3))
print("Cohen’s Kappa (VADER vs AFINN):", round(kappa_vader_afinn, 3))

Cohen’s Kappa (Rating vs VADER): 0.244
Cohen’s Kappa (Rating vs AFINN): 0.214
Cohen’s Kappa (VADER vs AFINN): 0.852


In [48]:
#  Cohen’s Kappa values.

def interpret_kappa(kappa):
    if kappa < 0.2:
        return "Poor agreement"
    elif kappa < 0.4:
        return "Fair agreement"
    elif kappa < 0.6:
        return "Moderate agreement"
    elif kappa < 0.8:
        return "Good agreement"
    else:
        return "Very strong agreement"

print("Rating vs VADER:", interpret_kappa(kappa_rating_vader))
print("Rating vs AFINN:", interpret_kappa(kappa_rating_afinn))
print("VADER vs AFINN:", interpret_kappa(kappa_vader_afinn))

Rating vs VADER: Fair agreement
Rating vs AFINN: Fair agreement
VADER vs AFINN: Very strong agreement
